# Data Acquisition — CSV, Excel, JSON and Parquet

## Required Packages 
* pandas
* openpyxl
* pyarrow
* SQLAlchemy
* requests
* beautifulsoup4
* lxml

## CSV 
Each row describes one student. SRN is an identifier, so preserve leading zeros.

In [1]:
import pandas as pd
df = pd.read_csv("data/students.csv")
print(df.shape)
display(df.head())
display(df.dtypes)

(13, 6)


,srn,name,program,city,enrolled_on,marks
0,1,Anu,MCA,Bengaluru,01-08-2026,85
1,2,Ravi,MCA,Mysuru,02-08-2026,92
2,3,Meera,BCA,Karur,03-08-2026,78
3,4,Kiran,MCA,Bengaluru,04-08-2026,68
4,5,Divya,MCA,Mysuru,05-08-2026,88


srn             int64
name           object
program        object
city           object
enrolled_on    object
marks           int64
dtype: object

## Reader options 

In [2]:
wrong = pd.read_csv("data/students_semicolon.csv")
right = pd.read_csv("data/students_semicolon.csv", sep=";", dtype={"srn":"string"}, parse_dates=["enrolled_on"])
print(wrong.head())
print(right.head())

  srn;name;program;city;enrolled_on;marks
0     001;Anu;MCA;Bengaluru;2026-08-01;85
1       002;Ravi;MCA;Mysuru;2026-08-02;92
2       003;Meera;BCA;Karur;2026-08-03;78
3   004;Kiran;MCA;Bengaluru;2026-08-04;68
4      005;Divya;MCA;Mysuru;2026-08-05;88
   srn   name program       city enrolled_on  marks
0  001    Anu     MCA  Bengaluru  2026-08-01     85
1  002   Ravi     MCA     Mysuru  2026-08-02     92
2  003  Meera     BCA      Karur  2026-08-03     78
3  004  Kiran     MCA  Bengaluru  2026-08-04     68
4  005  Divya     MCA     Mysuru  2026-08-05     88


## Excel sheets

In [3]:
students = pd.read_excel("data/students.xlsx", sheet_name="Students", dtype={"srn":"string"})
marks = pd.read_excel("data/students.xlsx", sheet_name="Marks", dtype={"srn":"string"})
display(students.head(), marks.head())

,srn,name,program,city,enrolled_on
0,001,Anu,MCA,Bengaluru,2026-08-01
1,002,Ravi,MCA,Mysuru,2026-08-02
2,003,Meera,BCA,Karur,2026-08-03
3,004,Kiran,MCA,Bengaluru,2026-08-04
4,005,Divya,MCA,Mysuru,2026-08-05


,srn,marks
0,001,85
1,002,92
2,003,78
3,004,68
4,005,88


## Merge
A left join retains every student. Validation rejects duplicate join keys.

In [4]:
joined = students.merge(marks,on="srn",how="left",validate="one_to_one")
print(joined.head())

   srn   name program       city enrolled_on  marks
0  001    Anu     MCA  Bengaluru  2026-08-01     85
1  002   Ravi     MCA     Mysuru  2026-08-02     92
2  003  Meera     BCA      Karur  2026-08-03     78
3  004  Kiran     MCA  Bengaluru  2026-08-04     68
4  005  Divya     MCA     Mysuru  2026-08-05     88


## JSON and JSON Lines 
JSON uses an array of records; JSONL stores one object per line.

In [5]:
flat_json = pd.read_json("data/students.json", dtype={"srn":"string"})
logs = pd.read_json("data/students.jsonl", lines=True, dtype={"srn":"string"})
display(flat_json.head(),logs.head())

,srn,name,program,city,enrolled_on,marks
0,001,Anu,MCA,Bengaluru,2026-08-01,85
1,002,Ravi,MCA,Mysuru,2026-08-02,92
2,003,Meera,BCA,Karur,2026-08-03,78
3,004,Kiran,MCA,Bengaluru,2026-08-04,68
4,005,Divya,MCA,Mysuru,2026-08-05,88


,srn,name,program,city,enrolled_on,marks
0,001,Anu,MCA,Bengaluru,2026-08-01,85
1,002,Ravi,MCA,Mysuru,2026-08-02,92
2,003,Meera,BCA,Karur,2026-08-03,78
3,004,Kiran,MCA,Bengaluru,2026-08-04,68
4,005,Divya,MCA,Mysuru,2026-08-05,88


## Nested JSON 
Flattening exposes nested keys while keeping lists inside cells.

In [6]:
import json
from pathlib import Path
records = json.loads(Path("data/students_nested.json").read_text(encoding="utf-8"))
flat = pd.json_normalize(records)
print(flat.columns.tolist())
display(flat.head(3))

['srn', 'name', 'skills', 'program.name', 'program.semester']


,srn,name,skills,program.name,program.semester
0,001,Anu,"[Python, SQL]",MCA,1
1,002,Ravi,"[Python, SQL]",MCA,1
2,003,Meera,"[Python, SQL]",BCA,1


## Repeated values 
After explode, one row represents a student–skill pair, rather than one student.

In [7]:
skills = flat.explode("skills",ignore_index=True)
display(skills.head())

,srn,name,skills,program.name,program.semester
0,001,Anu,Python,MCA,1
1,001,Anu,SQL,MCA,1
2,002,Ravi,Python,MCA,1
3,002,Ravi,SQL,MCA,1
4,003,Meera,Python,BCA,1


## Parquet 
Parquet carries column types. Select only the columns needed for analysis.

In [8]:
all_rows = pd.read_parquet("data/students.parquet")
small = pd.read_parquet("data/students.parquet", columns=["srn","marks"])
display(all_rows.head(),small.head())

,srn,name,program,city,enrolled_on,marks
0,001,Anu,MCA,Bengaluru,2026-08-01,85
1,002,Ravi,MCA,Mysuru,2026-08-02,92
2,003,Meera,BCA,Karur,2026-08-03,78
3,004,Kiran,MCA,Bengaluru,2026-08-04,68
4,005,Divya,MCA,Mysuru,2026-08-05,88


,srn,marks
0,001,85
1,002,92
2,003,78
3,004,68
4,005,88


## Validate 
Check schema, row count, missingness, duplicate rows and valid ranges.

In [9]:
df.tail()

,srn,name,program,city,enrolled_on,marks
8,9,Priya,BCA,Karur,09-08-2026,63
9,10,Hari,MCA,Bengaluru,10-08-2026,90
10,11,Sana,MCA,Mysuru,11-08-2026,101
11,12,Dev,BCA,Karur,12-08-2026,84
12,12,Dev,MCA,Karur,22-10-2026,-1


In [10]:
print("No of rows and Columns:", df.shape)
print("\n------------\n")
print("Data types of each column:\n", df.dtypes)
print("\n------------\n")
print("No of missing values:\n", df.isna().sum())
print("\n------------\n")
print("Duplicate rows:",df.duplicated().sum())
print("\n------------\n")
# Display records with invalid or missing marks
invalid = ~df["marks"].between(0, 100)
display(df.loc[invalid])
print("Number of invalid records:", invalid.sum())
print("\n------------\n")
print(df["srn"].unique())
print(df["srn"].nunique())
assert df["srn"].is_unique, "Duplicate SRNs found!"

No of rows and Columns: (13, 6)

------------

Data types of each column:
 srn             int64
name           object
program        object
city           object
enrolled_on    object
marks           int64
dtype: object

------------

No of missing values:
 srn            0
name           0
program        0
city           0
enrolled_on    0
marks          0
dtype: int64

------------

Duplicate rows: 0

------------



,srn,name,program,city,enrolled_on,marks
10,11,Sana,MCA,Mysuru,11-08-2026,101
12,12,Dev,MCA,Karur,22-10-2026,-1


Number of invalid records: 2

------------

[ 1  2  3  4  5  6  7  8  9 10 11 12]
12


AssertionError: Duplicate SRNs found!

## Export 
Outputs go in a separate folder. Raw inputs remain unchanged.

In [11]:
joined.to_csv("out/students_joined.csv",index=False)
joined.to_parquet("out/students_joined.parquet",index=False)
print("Saved CSV and Parquet results")

Saved CSV and Parquet results
